In [264]:
import numpy as np
import sys
sys.path.append("../../")
sys.path.append("../../Visualization/")
sys.path.append("../../../")
import inflation
import igl
from periodic_simulation_setup import *
import json

import parallelism, multiprocessing, itertools, setproctitle
import os, time, numpy as np

import json

In [267]:

# time = '2024_01_14_11_09'
# name = 'square_with_ellipse_hole_angle_height_1.6_width_1'


time = '2024_01_17_11_33'
name = 'square_with_ellipse_hole_angle_width_height'
experiment_file = '../../experiments/parallelized_experiments/output/{}/{}/experiment_result.json'.format(name, time)
stiffness_path = '../../experiments/parallelized_experiments/output/{}/{}'.format(name, time)

In [268]:
import MeshFEM, visualization

In [269]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [272]:
angles = np.linspace(0, 45, 16)[2:]
widths = np.linspace(0.1, 0.9, 9)
heights = np.linspace(1.3, 1.8, 6)

In [328]:
angles

In [281]:
label = "{:.2f}_{:.2f}_{:.2f}".format(angles[0], widths[2], heights[5])

In [282]:
m = MeshFEM.mesh.Mesh('../../experiments/parallelized_experiments/output/{}/{}/{}/mesh_{}_{}.obj'.format(name, time, label, name, label))
fusing_vtx = np.load('../../experiments/parallelized_experiments/output/{}/{}/{}/fusedVtx_{}_{}.npy'.format(name, time, label, name, label))

visualization.plot_2d_mesh(m, pointList=fusing_vtx, width=5, height=5)

In [283]:
vertices = m.vertices()
vertices = np.concatenate((vertices, np.zeros((len(vertices), 1))), axis = 1)
m = MeshFEM.mesh.Mesh(vertices, m.elements())
# fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusing_vtx, epsilon = 1e-9)
ipu.setVars(np.load('../../experiments/parallelized_experiments/output/{}/{}/{}/{}_dofs_before_stiffness_{}.npy'.format(name, time, label, name, label)))

In [284]:
viewer = TriMeshViewer(ipu, width=500, height=500)
viewer.showWireframe(False)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
viewer.show()

In [285]:
# experiment_file = '../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15/experiment_result.json'

# stiffness_path = '../../experiments/parallelized_experiments/output/double_zigzag_dash/2024_01_08_18_15'
# name = 'double_zigzag_dash'

### Overview

In [286]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

In [287]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [288]:
df = pd.DataFrame(data['data'])
fig, axes = plt.subplots(nrows = 1, ncols = 3, figsize = (20, 5))
a = (df.hist('Ipu simulation succeed', ax = axes[0]), df.hist('Planar equilibrium', ax = axes[1]), df.hist('Simulation Kappa value', ax = axes[2]))

In [289]:
import visualize_stiffness
import importlib
importlib.reload(visualize_stiffness)

In [290]:
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [291]:
invalid_tags = np.array(df['name'][df['Planar equilibrium'] != 1])

In [292]:
invalid_tags

In [293]:
df['Simulation Kappa value'][np.array(df['Planar equilibrium']) != 1]

In [294]:
kappa_path = None

In [295]:
# if valid_tags[0] == '0.50':
#     valid_tags = valid_tags[3:]
parameters = angles = np.array(data['pattern_parameters'][0]['values'])

In [296]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [297]:
(x_scale_factors - y_scale_factors)[np.where(x_scale_factors < y_scale_factors)[0]]

In [298]:
valid_tags[np.where(x_scale_factors < y_scale_factors)[0]]

In [299]:
np.max(y_scale_factors), np.argmax(y_scale_factors)

In [301]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = False)

In [302]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [303]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [304]:
(x_scale_factors - y_scale_factors)[np.where(x_scale_factors < y_scale_factors)[0]]

In [305]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [306]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

In [307]:
np.argmax(min_bending_stiffness)

In [308]:
np.argmin(min_bending_stiffness)

In [311]:
samples = np.array([[float(n) for n in tag.split('_')[:]] for tag in valid_tags])

In [312]:
plt.scatter(samples[:, 0], samples[:, 1])

In [325]:
def show_tags(threshold = 0):
    curr_tags = valid_tags[(np.where(min_bending_stiffness > threshold))]
    samples = np.array([[float(n) for n in tag.split('_')[:]] for tag in curr_tags])
    print(curr_tags)
    plt.scatter(samples[:, 0], samples[:, 1], alpha = 0.3)

In [326]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [327]:
interact(show_tags, threshold=widgets.FloatSlider(min=-0.1, max=2, step=0.1, value=0));

### Get scale function convex hull

In [137]:
import matplotlib.cm as cm
import matplotlib as mpl

In [138]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

In [139]:
hull

In [142]:
import matplotlib.pyplot as plt
plt.plot(points[:,0], points[:,1], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')
plt.plot(points[hull.vertices,0], points[hull.vertices,1], 'r--', lw=2)
plt.plot(points[hull.vertices[0],0], points[hull.vertices[0],1], 'ro')
plt.show()

In [143]:
import numpy.linalg as la

# Need to plot the patches over the min and max scale factors, so we can get the polygon that constrain the singular values
# The scale factors we are considering during the parametrization are from the flattening, so it's the change from the inflated state to the fabricated state, hence we need to take one over the factors we have from the average deformation gradient from homogenization.
# max_scale_factor = 1 / np.array(scale_factor_data)[:, 0]
# min_scale_factor = 1 / np.array(scale_factor_data)[:, 1]

fig, ax = plt.subplots(figsize = (10, 10))

# plt.scatter(x_scale_factor, y_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)
# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.3)

# plt.scatter(y_scale_factor, x_scale_factor, label = data_info[i][1], s = 50, alpha = 0.8, c = min_stiffness)


points = np.concatenate((x_scale_factors.reshape((-1, 1)), y_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

for simplex in hull.simplices:
    plt.plot(points[simplex, 0], points[simplex, 1], 'k-')

ax.title.set_text("Scale factors")
plt.xlabel("x scale factors")
plt.ylabel("y scale factors")

plt.scatter(x_scale_factors, y_scale_factors, label = 'min_stiffness', s = 200, alpha = 1, c = min_bending_stiffness)
# plt.scatter(x_scale_factors, y_scale_factors, label = 'max_stiffness', s = 200, alpha = 1, c = max_bending_stiffness)

# Plot x = y line
lims = [
np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, 'k-', alpha=0.75, zorder=0)
ax.set_aspect('equal')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend()
fig.tight_layout()
plt.savefig('scale_factor_values_{}.png'.format(name), dpi = 300)

### Validate the max and min scale factors are aligned with the x and y axis

In [144]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [145]:
eqns = hull.equations

In [146]:
import parametrization_helper, importlib
importlib.reload(parametrization_helper)

In [147]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [148]:
import visualize_stiffness
importlib.reload(visualize_stiffness)

In [149]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [150]:
plt.plot(stiffness_coefficients[:, 4])

In [151]:
np.set_printoptions(suppress=True, precision=4)

In [152]:
np.argmax(stiffness_coefficients[:, 1]), np.argmax(stiffness_coefficients[:, 2])

In [153]:
def get_stiffness_polynomial(s, theta):
    return s[0] * np.cos(theta)**2 * np.sin(theta)**2 + s[1] * np.cos(theta)**3 * np.sin(theta) + s[2] * np.cos(theta) * np.sin(theta)**3 + s[3] * np.cos(theta)**4 + s[4] * np.sin(theta)**4

In [154]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

In [155]:
grid_data = np.zeros((9, len(parameters)))

for i in range(len(valid_tags)):
    grid_data[0][i] = max_scale_factors[i]
    grid_data[1][i] = min_scale_factors[i]
    grid_data[2][i] = x_scale_factors[i]
    grid_data[3][i] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][i] = stiffness_coefficients[i][s]

In [156]:
importlib.reload(parametrization_helper)

In [55]:
grid_data.shape

In [56]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, (parameters))

In [57]:
grid_data.shape

In [58]:
scale_factors_grid_data = np.zeros((2, len(parameters)))
for i in range(len(parameters)):
    scale_factors_grid_data[0][i] = x_scale_factors[i]
    scale_factors_grid_data[1][i] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, (parameters))

In [59]:
test_parameters = np.linspace(0, 45, 100)

In [60]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
# titles = ['max scale factors', 'min scale factors', 's1', 's2', 's3', 's4', 's5']

for i in range(9):
    axes[i].plot(test_parameters, splines[i * 3 + 0](test_parameters))
    axes[i].set_title(titles[i], fontsize=21)

In [61]:
stiffness_coefficients = np.array(stiffness_coefficients)

In [62]:
stiffness_coefficients.shape

In [63]:
fig, axes = plt.subplots(1, 9, figsize=(45, 8))
titles = ['max scale factors', 'min scale factors', 'x scale factors', 'y scale factors', 's1', 's2', 's3', 's4', 's5']
data = [max_scale_factors, min_scale_factors, x_scale_factors, y_scale_factors, stiffness_coefficients[:, 0], stiffness_coefficients[:, 1], stiffness_coefficients[:, 2], stiffness_coefficients[:, 3], stiffness_coefficients[:, 4]]

for i in range(9):
    axes[i].plot(parameters, data[i])
    axes[i].set_title(titles[i], fontsize=21)

### Parametrization

In [64]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [65]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [66]:
from periodic_simulation_setup import *

In [67]:
import parametrization_optimization_helper

In [68]:
import utils, mesh_utilities
importlib.reload(utils)

In [69]:
target_surf = mesh.Mesh("../../../../examples/igloo.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)

In [70]:
lines = np.array(eqns)

### New local global with convex hull

In [71]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)
lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [72]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [73]:
importlib.reload(visualization)
visualization.visualize_both(lg, show_main = True)

In [74]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [75]:
default_pattern_params = np.array([30]  * len(lg.getAlphas()))

In [76]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[10, 45]])
rparam.patternParamNormalizationFactors = np.array([1])
rparam.diffRegW = 0.0

In [77]:
visualization.visualize_both(rparam, height = 4)

In [78]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [79]:
num_pattern_params = 1

In [80]:
parametrization_optimization_helper.initialize_pattern_parameters(rparam, lines, num_pattern_params)

In [81]:
phiRegW = parametrization_optimization_helper.add_phi_regularization(rparam, lines, num_pattern_params)

In [82]:
patternRegW = parametrization_optimization_helper.add_pattern_regularization(rparam, lines, num_pattern_params, phiRegW)

### Bending

In [ ]:
bendRegW = parametrization_optimization_helper.add_bending_energy(rparam, lines, num_pattern_params, phiRegW, patternRegW)

## Upsampling and channel generation

In [93]:
import parametrization_helper
importlib.reload(parametrization_helper)

In [116]:


def sample_ellipse_evenly(h, k, iw, ih, num_points):
    # Eccentricity of the ellipse
    e = np.sqrt(1 - (min(iw, ih) / max(iw, ih))**2)
    # Eccentric anomaly (angles for the points)
    E = np.linspace(0, 2*np.pi, num_points)
    # True anomaly (angles adjusted for the eccentricity)
    t = 2*np.arctan(np.sqrt((1+e)/(1-e)) * np.tan(E/2))
    # Adjust the angles to be in the range [0, 2pi]
    t = np.mod(t, 2*np.pi)
    # Compute the points on the ellipse
    x = h + iw * np.cos(t)
    y = k + ih * np.sin(t)
    return x, y

In [202]:
import numpy as np

def fusing_curve_polyline(patternParams):
    # Draw dash_line
    angle = patternParams[0]
    theta = np.radians(angle)  # convert angle to radians

    iw = 0.1
    ih = height    
    
    # Center of the ellipse
    h = k = 0

    # Number of points in the fusing line
    num_points = 20

    # Compute the ratio of the width and height
    ratio = iw / ih

    # Parameter values
    t = np.linspace(0, 2*np.pi, num_points)
    
    print(t * 180 / np.pi)
    # Adjust the angles based on the ratio
    t_adjusted = np.arctan2(ratio * np.sin(t), np.cos(t))
    t_adjusted = np.mod(t_adjusted, 2*np.pi)
    print(t_adjusted * 180 / np.pi)

    # Compute the points on the ellipse
    x_unrotated = iw * np.cos(t_adjusted)
    y_unrotated = ih * np.sin(t_adjusted)

    # Rotate the ellipse by angle
    x = h + (x_unrotated) * np.cos(theta) - (y_unrotated) * np.sin(theta)
    y = k + (x_unrotated) * np.sin(theta) + (y_unrotated) * np.cos(theta)

    # Combine x and y into an array of points
    points = np.column_stack((x, y))
    return [(points + np.array([2.5, 2.5])) / 5 * np.pi]

In [296]:
import numpy as np
from scipy import optimize
from scipy.special import ellipe
from scipy.special import ellipeinc

def fusing_curve_polyline(patternParams):
    # Draw dash_line
    angle = patternParams[0]
    theta = np.radians(angle)  # convert angle to radians

    iw = width
    ih = height    
    
    # Center of the ellipse
    h = k = 0

    # Number of points in the fusing line
    num_points = 20

    # Eccentricity of the ellipse
    e = np.sqrt(1 - (min(iw, ih) / max(iw, ih))**2)

    # Total arc length of the ellipse
    total_arclength = 4 * max(iw, ih) * ellipe(e**2)

    # Compute evenly spaced arc lengths
    arc_lengths = np.linspace(0, total_arclength, num_points)

    # Function to compute the difference between the target and actual arc length
    def func(t, target_arclength, e):
        return target_arclength - max(iw, ih) * ellipeinc(t, e**2)

    # Compute the angles for the points
    t = np.array([optimize.root(func, [0], args=(l, e)).x[0] for l in arc_lengths])

    # Compute the points on the ellipse
    x_unrotated = iw * np.cos(t)
    y_unrotated = ih * np.sin(t)

    # Rotate the ellipse by angle
    x = h + (x_unrotated) * np.cos(theta) - (y_unrotated) * np.sin(theta)
    y = k + (x_unrotated) * np.sin(theta) + (y_unrotated) * np.cos(theta)

    # Combine x and y into an array of points
    points = np.column_stack((x, y))
    return [(points + np.array([2.5, 2.5])) / 5 * np.pi]

In [297]:
points = fusing_curve_polyline([45])

In [298]:
# fusing_curve_polyline([2, 60])

In [299]:
fusing_lines = fusing_curve_polyline([36])[0].reshape(-1, 2)
fusing_edges = [[i, i + 1] for i in range(len(fusing_lines) - 1)]

In [300]:
boundary_vertices = [[0, 0], [np.pi, 0], [np.pi, np.pi], [0, np.pi]]

In [301]:
boundary_edges = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) + len(fusing_lines)

In [302]:
visualization.plot_line_segments(list(fusing_lines) + boundary_vertices, fusing_edges + list(boundary_edges))
plt.scatter(np.array(fusing_lines)[:, 0], np.array(fusing_lines)[:, 1])

In [263]:
sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines, boundaryVxs, boundaryEdges, upsampleMesh_vertices,  upsampleMesh_triangles, upsampledAngles, upsampledPatternParams = parametrization_helper.get_polyline_from_pattern_parameters(rparam, fusing_curve_polyline, nsubdiv = 3, frequency=0.1, duplicates_removable_threshold=[1e-4, 1e-2, 1e-1, 1e0, 2e0, 3e0])

In [271]:

#### Get raw boundary edges
raw_flatten_mesh = MeshFEM.mesh.Mesh(upsampleMesh_vertices, upsampleMesh_triangles)
raw_boundaryLoop = raw_flatten_mesh.boundaryElements()
raw_boundaryVxsIdxs = raw_flatten_mesh.boundaryVertices()
raw_boundaryVxs = upsampleMesh_vertices[raw_boundaryVxsIdxs]

In [272]:
if len(boundaryEdges) > 1:
    concatenated_boundary_edges = []
    for polyline in boundaryEdges:
        concatenated_boundary_edges.extend(polyline)
    concatenated_boundary_edges = np.array(concatenated_boundary_edges)
else:
    concatenated_boundary_edges = np.array(boundaryEdges[0])

In [273]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)
visualization.plot_line_segments(sheet_vxs, concatenated_polylines, width = 5, height = 5)
visualization.plot_line_segments(list(sheet_vxs) + list(boundaryVxs), list(concatenated_polylines) + list(concatenated_boundary_edges + len(sheet_vxs)), width = 5, height = 5)
plt.scatter(boundaryVxs[concatenated_boundary_edges[:, 0], 0], boundaryVxs[concatenated_boundary_edges[:, 0], 1], c = np.arange(len(boundaryVxs)), cmap = mpl.colormaps['Greys'])

## Meshing and inflation simulation

In [296]:
import mesher_helper
importlib.reload(mesher_helper)

In [297]:
import time
time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [298]:
importlib.reload(parametrization_helper)

In [299]:
selected_elements = [np.array(sublist)[:, 0] for sublist in boundaryEdges[1:]]
holes_vxs = list(boundaryVxs[selected_elements])

threshold = 4
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    
    if len(polyline) < threshold:
        continue
    # Check if the polyline forms a closed loop
    if polyline[0, 0] == polyline[-1, 1]:
        # If it's a closed loop, append it to holes_vxs
        curr_points = np.array(sheet_vxs[polyline[:, 0]])[:, :2]
        line = LineString(curr_points)

        # Check if the line has self-intersections
        if not line.is_simple:
            hull = ConvexHull(curr_points)  # We only consider the first two columns (x, y coordinates)
            hull_points = np.concatenate((curr_points[hull.vertices], np.zeros((len(hull.vertices), 1))), axis = 1)

            holes_vxs.append(hull_points[:, :3])
        else:
            holes_vxs.append(np.array(sheet_vxs[polyline[:, 0]]))
    else:
        # If it's not a closed loop, connect the end points with a half circular arc that points away from the polyline
        # Convert polyline to 2D points
        points = np.array(sheet_vxs[polyline[:, 0]])

        # Calculate the midpoint, direction, and length of the last segment
        p1 = points[0]
        p2 = points[-1]
        midpoint = (p1 + p2) / 2
        # Calculate the radius and center of the semi-circle
        radius = np.linalg.norm(p2 - p1) / 2
        center = midpoint

        # Calculate the direction from p1 to p2
        direction_vector = p2 - p1
        direction = np.arctan2(direction_vector[1], direction_vector[0])
        # Calculate the middle point of the points
        middle_point = points[len(points) // 2]

        # Calculate the vector from the first point to the middle point
        vector = middle_point - points[0]

        # Calculate the z-component of the cross product of the direction vector and the vector
        cross_product_z = direction_vector[0] * vector[1] - direction_vector[1] * vector[0]


        # Generate points on the semi-circle from p2 to p1
        if cross_product_z > 0:
            t = np.linspace(direction, direction - np.pi, 6)[1:-1]
        else:
            t = np.linspace(direction, direction + np.pi, 6)[1:-1]
        semi_circle_points = np.empty((4, 3))
        semi_circle_points[:, 0] = center[0] + radius * np.cos(t)
        semi_circle_points[:, 1] = center[1] + radius * np.sin(t)
        semi_circle_points[:, 2] = 0
        # Append the polyline points and the semi-circle points to holes_vxs
        final_points = np.concatenate([points, semi_circle_points])
        new_points, _ = parametrization_helper.remove_duplicates(final_points, [], 1e0)
        if (len(new_points) < threshold):
            continue
        hull = ConvexHull(new_points[:, :2])  # We only consider the first two columns (x, y coordinates)
        hull_points = np.concatenate((new_points[hull.vertices], np.zeros((len(hull.vertices), 1))), axis = 1)
        holes_vxs.append(hull_points[:, :3])

In [300]:
importlib.reload(mesher_helper)

In [301]:
plot_points = []
plot_edges = []
for polyline in holes_vxs:
    plot_edges.extend(np.array([[i, (i+1)%len(polyline)] for i in range(len(polyline))]) + len(plot_points))
    plot_points.extend(polyline)


In [302]:
visualization.plot_line_segments(plot_points, plot_edges, width = 5, height = 5)

In [303]:
importlib.reload(mesher_helper)

In [304]:
v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(4, boundaryVxs[np.array(boundaryEdges[0])[:, 0]], holes_vxs, [], [], gui = False)

In [ ]:
import numpy as np
import copy

# Use the function
new_v, new_f, new_fusing_without_boundary = parametrization_helper.remove_dangling_vertices(v, f - 1, fusing_data)
m = MeshFEM.mesh.Mesh(new_v, new_f)
new_fusing = copy.copy(new_fusing_without_boundary)
new_fusing[m.boundaryVertices()] = True

In [ ]:
fusing_data, new_fusing

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
nonmanifold_vxs = parametrization_helper.get_non_manifold_boundary_vertices(m)
print(nonmanifold_vxs)

In [217]:
# m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, SV, SE, triArea=1e0)


In [218]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(new_fusing) == 1)[0], width=10, height=10)


In [221]:
import inflation
isheet = inflation.InflatableSheet(m, new_fusing)

### Save pattern

In [222]:
polylines = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    polylines.append(sheet_vxs[np.array(list(polyline[:, 0]) + list([polyline[-1, 1]]))][:, :2].tolist())
parametrization_helper.save_to_obj(boundaryVxs[np.array(boundaryEdges)[0][:, 0]], polylines, 'igloo_{}_sheet_pattern_{}_margin_{}.obj'.format(name, time_stamp, 0))

### End

In [223]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [224]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [225]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [226]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [227]:
import boundaries
bdryVars = boundaries.getOuterBoundaryVars(isheet)
fixedVars = bdryVars

In [228]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

# fixedVars, hessianShift = bdryVars, 1e-6
fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [229]:
isheet.pressure = 1e-5

In [230]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [231]:
isheet.pressure = 5e-2

In [232]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [233]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [ ]:
import gzip

In [290]:
pickle.dump(isheet,  gzip.open("igloo_pattern_optimized_{}_low_frequency_with_bending_high_resolution.pkl.gz".format(time_stamp), 'wb'))

### Generate Fabrication Files

In [ ]:
old_to_new = np.arange(np.max(isheet.wallVertices()) + 1)

In [ ]:
old_to_new[isheet.wallVertices()] = np.arange(len(isheet.wallVertices()))

In [ ]:
from parametrization_helper import form_polylines

In [ ]:
result_vxs = isheet.restWallVertexPositions()
result_edges = old_to_new[isheet.wallBoundaryEdges()]
result_edges = form_polylines(result_edges.tolist())
concatenated_polylines = []
for polyline in result_edges:
    concatenated_polylines.extend(polyline)


In [ ]:
visualization.plot_line_segments(result_vxs, concatenated_polylines)